# GNN Flood Forecasting Model
Predicts streamflow 24 hours ahead at 37 primary gauge sites.

**Two data sources are used:**
- `raw-streamflow-15min` — for computing edge attributes (lag, scale, R²) via `analyze_gauge_relationship`
- `flood-dataset-top49`  — hourly feature-rich dataset for model training (all 49 sites)

Edge attributes are computed once from the 15-min parquet, then frozen for training.

In [1]:
import json
import os
import numpy as np
import polars as pl
import torch
import wandb
from torch.utils.data import TensorDataset, DataLoader

from src.preprocessing.gnn_preprocessing import gnn_processor, get_all_sites
from src.models.gnn_model import StreamflowGNN

c:\Users\mroth\Comp 549\Flood-Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load upstream pair dict

In [3]:
with open("../../src/utils/upstream_pair_dict.json") as f:
    upstream_pair_dict = json.load(f)

all_sites, primary_sites, upstream_only = get_all_sites(upstream_pair_dict)
print(f"Primary        : {len(primary_sites)}")
print(f"Upstream-only  : {len(upstream_only)}")
print(f"Total          : {len(all_sites)}")
print(f"\nUpstream-only site IDs: {upstream_only}")

Primary        : 37
Upstream-only  : 12
Total          : 49

Upstream-only site IDs: ['06027600', '06062500', '06091700', '06209500', '06211500', '06326500', '06449500', '06790500', '06799000', '06803486', '06820410', '06880800']


## 2. Config

In [4]:
STATIC_FEATURES = [
    "longitude", "latitude", "DRAIN_SQKM", "artificial_path_pct",
    "wb5100_ann_mm", "snw_pc_syr", "snow_ice_nlcd06", "barren_nlcd06",
    "mains100_plant", "hga", "hgc", "bulk_density_avg", "elev_max_m", "aspect_deg"
]

DYNAMIC_FEATURES = [
    "streamflow_cfs_mean", "streamflow_cfs_max", "streamflow_cfs_min",
    "gage_height_ft_mean",
    "precipitation_mm",
    "temperature_c",
    "potential_evaporation_mm",
    "specific_humidity_kgkg",
    "shortwave_radiation_wm2",
    "longwave_radiation_wm2",
    "wind_speed_ms",
    "surface_pressure_pa",
    "cape_jkg",
    "convective_precip_fraction",
]

WINDOW_SIZE = 72

config = {
    "input_cols":         DYNAMIC_FEATURES + STATIC_FEATURES,
    "static_cols":        STATIC_FEATURES,
    "target":             "streamflow_cfs_target_24h",
    "train_split":        0.8,
    "val_split":          0.9,
    "lag_window":         1,
    "frequency":          "hourly",
    "split_time_days":    30,
    "site_scaling":       False,
    # *** UPDATE: point to 49-site hourly artifact ***
    "file_path":          "flood-dataset-top49",
    "file_name":          "flood_model_top49",
    # GNN-specific
    "upstream_pair_dict": upstream_pair_dict,
}

## 3. Download raw 15-min parquet (for edge attributes)

Edge attributes are computed from `raw-streamflow-15min` — the same artifact
used in `top_37_upstream_pair.ipynb`. This only needs to run once.

In [5]:
api = wandb.Api()
artifact = api.artifact("flood-forecasting/raw-streamflow-15min:latest")
artifact_dir = artifact.download()
parquet_path = os.path.join(artifact_dir, "streamflow_15min", "*.parquet")
print(f"Parquet path: {parquet_path}")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\mroth\.netrc.
wandb: Downloading large artifact 'raw-streamflow-15min:latest', 1467.48MB. 20 files...
wandb:   20 of 20 files downloaded.  
Done. 00:00:02.2 (653.9MB/s)


Parquet path: c:\Users\mroth\Comp 549\Flood-Forecasting\src\models\artifacts\raw-streamflow-15min-v0\streamflow_15min\*.parquet


## 4. Build gnn_processor and compute edge attributes

**Order matters:**
1. `compute_edge_attributes()` — runs `analyze_gauge_relationship` for all 16 valid pairs
2. `pull_wandb()` — loads the hourly flood dataset for all 49 sites and builds the graph

In [6]:
gnn_pcr = gnn_processor(config)

# Step 1: compute lag_hours, scale_m, intercept_b, r_squared for each edge
# Uses raw 15-min parquet — same logic as top_37_upstream_pair.ipynb
gnn_pcr.compute_edge_attributes(
    parquet_path     = parquet_path,
    max_lag_hours    = 72,
    timestep_minutes = 15,
)

[gnn_processor] Total sites   : 49
[gnn_processor] Primary       : 37
[gnn_processor] Upstream-only : 12
Computing edge attributes for 16 upstream pairs...

Upstream     → Primary        lag_h      R²
------------------------------------------------
06803495     → 06803500        0.00  0.9951
06447500     → 06446700        0.00  0.5679
06790500     → 06793000        0.75  0.6410
06820410     → 06820500       16.50  0.6236
06803486     → 06803495        0.50  0.6470
06027600     → 06036650        2.25  0.9894
06209500     → 06207500        3.00  0.7270
06803500     → 06803513        1.25  0.9746
06062500     → 06061500        6.50  0.8137
06799000     → 06799315        2.00  0.7956
06880800     → 06881000        8.00  0.6906
06917000     → 06917060       21.25  0.6279
06211500     → 06211000        0.00  0.5409
06326500     → 06308500        2.75  0.6549
06091700     → 06093200        4.00  0.8803
06449500     → 06450500        0.00  0.6136

Edge attributes ready for 16/16 pairs.


In [7]:
# Step 2: load hourly flood dataset for all 49 sites; graph is built automatically
gnn_pcr.pull_wandb()
gnn_pcr.summary()

CommError: artifact membership 'flood-dataset-top49:latest' not found in 'connorjsmith28-rice-university/flood-forecasting'

In [ ]:
# Inspect the edge attributes
gnn_pcr.edge_attr_summary()

## 5. Build tensors and DataLoaders

In [ ]:
print("Building tensors...")
X_train, y_train = gnn_pcr.build_gnn_tensors("train", WINDOW_SIZE)
X_val,   y_val   = gnn_pcr.build_gnn_tensors("val",   WINDOW_SIZE)
X_test,  y_test  = gnn_pcr.build_gnn_tensors("test",  WINDOW_SIZE)

print(f"Train X: {X_train.shape}  y: {y_train.shape}")
print(f"Val   X: {X_val.shape}")
print(f"Test  X: {X_test.shape}")

In [ ]:
# Static features: one row per node, constant across time
static_features_list = []
for site in gnn_pcr.ordered_sites:
    row = (
        gnn_pcr.train_X_scaled
        .filter(pl.col("site_id") == site)
        .select(STATIC_FEATURES)
        .head(1)
        .to_numpy()
    )
    static_features_list.append(row[0])

static_features = torch.tensor(np.array(static_features_list), dtype=torch.float32)
print(f"Static features: {static_features.shape}  [num_nodes, num_static_features]")

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False)

## 6. Build model

Note `edge_attr` with shape `[num_edges, 4]` is now passed to `GATv2Conv` via `edge_dim=4`.
The model signature requires `edge_attr` in every forward pass.

In [ ]:
graph = gnn_pcr.get_graph()

model = StreamflowGNN(
    num_dynamic_features = len(DYNAMIC_FEATURES),
    num_static_features  = len(STATIC_FEATURES),
    hidden_dim  = 64,
    lstm_layers = 2,
    gat_heads   = 4,
    gat_layers  = 2,
    dropout     = 0.2,
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

In [ ]:
# Smoke test — forward pass on single sample
pred = model(
    x_seq        = X_train[0],          # [num_nodes, window_size, dyn_features]
    x_static     = static_features,
    edge_index   = graph.edge_index,
    edge_attr    = graph.edge_attr,     # [num_edges, 4]  ← lag_hours, scale_m, intercept_b, r_squared
    primary_mask = graph.primary_mask,
)
print(f"Output shape: {pred.shape}")   # [37, 1]

## 7. Train

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

model.fit(
    train_loader    = train_loader,
    val_loader      = val_loader,
    edge_index      = graph.edge_index,
    edge_attr       = graph.edge_attr,      # physical edge features
    primary_mask    = graph.primary_mask,
    static_features = static_features,
    optimizer       = optimizer,
    epochs          = 50,
    patience        = 5,
)

In [ ]:
model.plot_training_history()

## 8. Evaluate

In [ ]:
metrics = model.evaluate(
    test_loader        = test_loader,
    edge_index         = graph.edge_index,
    edge_attr          = graph.edge_attr,
    primary_mask       = graph.primary_mask,
    static_features    = static_features,
    target_scaler      = gnn_pcr.target_scaler,
    primary_site_order = sorted(gnn_pcr.primary_sites),
)

## 9. Inspect attention weights (interpretability)

The attention weight for each edge tells you how much the GNN weighted that
upstream gauge when updating each downstream node. Edges with high `r_squared`
and physically appropriate `lag_hours` should show higher weights.

In [ ]:
alpha = model.get_edge_attention_weights(
    x_seq        = X_test[0],
    x_static     = static_features,
    edge_index   = graph.edge_index,
    edge_attr    = graph.edge_attr,
    primary_mask = graph.primary_mask,
    layer        = 0,
)

idx_to_site = {v: k for k, v in gnn_pcr.node_id_to_index.items()}
ea = graph.edge_attr
ei = graph.edge_index

print(f"\n{'Upstream':<12} {'Primary':<12} {'R²':>6}  {'lag_h':>6}  {'Attn (mean heads)':>18}")
print("-" * 64)
for i in range(ei.shape[1]):
    up   = idx_to_site[ei[0, i].item()]
    dn   = idx_to_site[ei[1, i].item()]
    r2   = ea[i, 3].item()
    lag  = ea[i, 0].item()
    attn = alpha[i].mean().item()
    print(f"{up:<12} {dn:<12} {r2:>6.4f}  {lag:>6.2f}  {attn:>18.6f}")

## 10. Save

In [ ]:
torch.save({
    "model_state_dict":  model.state_dict(),
    "node_id_to_index":  gnn_pcr.node_id_to_index,
    "primary_sites":     gnn_pcr.primary_sites,
    "upstream_only":     gnn_pcr.upstream_only,
    "edge_index":        graph.edge_index,
    "edge_attr":         graph.edge_attr,
    "primary_mask":      graph.primary_mask,
    "config":            {k: v for k, v in config.items() if k != "upstream_pair_dict"},
}, "gnn_flood_model.pt")

print("Saved to gnn_flood_model.pt")